In [123]:
# packages

import requests
import json
import time
import datetime
import os
import pandas as pd
#import google.generativeai as genai
import google.genai
import io
try: # google colab не запускается, когда раним через workflow, он там есть по умолчанию, поэтому имени в PyPL такого нет
    from google.colab import userdata, drive
except ImportError:
    userdata = None
    drive = None
from datetime import date, timedelta, datetime
from typing import List
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from google.oauth2.service_account import Credentials
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaIoBaseDownload, MediaIoBaseUpload
from pydantic import BaseModel

In [124]:
#%pip install -r "requirements.txt"


In [125]:
#Defaulting to user installation because normal site-packages is not writeable
#%pip install google.genai
#%pip install google-api-python-client
#%pip show google-genai

In [126]:
USE_SANDBOX = False  # Set to True to use sandbox folders
folders = 'folders_sandbox.txt' if USE_SANDBOX else 'folders_main.txt'
print("Folders:", "SANDBOX (FOLDERS_SANDBOX)" if USE_SANDBOX else "MAIN (FOLDERS_MAIN)")

Folders: MAIN (FOLDERS_MAIN)


In [127]:
import os
import json
from pathlib import Path

FOLDERS_FILE = Path("keys") / folders
with open(FOLDERS_FILE, "r", encoding="utf-8") as f:
    folder = json.load(f)

if not folder:
    raise ValueError("There are no folder links (check the file or environment variable)!")

In [128]:
# Auxilliary
#HEADERS = {"User-Agent": "Mozilla/5.0"}

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/121.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;"
        "q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8"
    ),
    "Accept-Language": "ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "DNT": "1",
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "none",
    "Sec-Fetch-User": "?1",
}





## Google Drive access (Google Cloud)

**Console:** [console.cloud.google.com](https://console.cloud.google.com)

---

### Option A — OAuth (Desktop, local runs)

#### 1. Create an OAuth client

1. **APIs & Services** → **Credentials** → **OAuth Client ID**  
   (if no client exists yet, configure the consent screen first)
2. Type: **Desktop app** → **Client secrets** → **Download JSON**  
   ⚠️ The secret JSON is available **only once** when you create the client.

Guide: [Using Python with Google APIs](https://sky.pro/wiki/media/kak-ispolzovat-python-dlya-raboty-s-api-google/)

#### 2. OAuth consent screen

1. **APIs & Services** → **OAuth consent screen** → **Audience**
2. **Publishing status:** *Testing*
3. Add yourself under **Test users**

Docs: [Brand verification (dev/test)](https://developers.google.com/identity/protocols/oauth2/production-readiness/brand-verification?hl=en#projects-used-in-dev-test-stage)

#### 3. Obtain a token

Run in order: `1auth.py` → `2encode_token.py`

---

### Option B — Service account (server / workflow)

The **next cell** loads a **service account** JSON file, not an OAuth client secret.

Detailed guide: [Drive access setup (iSpring)](https://docs.ispring.ru/pages/viewpage.action?pageId=119866165)

> ⚠️ After creating the key, grant the service account **edit access to the Drive folder** (share the folder with the `...@....gserviceaccount.com` email).

#### Steps in the console

| Step | Action |
|------|--------|
| 1 | **IAM & Admin** → **Service Accounts** |
| 2 | **Create service account** |
| 3 | Grant **Google Drive API** access |
| 4 | **Keys** → **Add Key** → **JSON** — download the file |

Example (Calendar scope; this project uses a different scope for Drive):

```python
from google.oauth2 import service_account

SCOPES = ["https://www.googleapis.com/auth/calendar.readonly"]
SERVICE_ACCOUNT_FILE = "/path/to/service.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES
)
```

The next cell uses scope `https://www.googleapis.com/auth/drive` for Drive.


In [129]:

with open("keys/service_account.json", 'r', encoding='utf-8') as f: # to run it locally, put it in content
#service account credentials JSON file (created and downloaded from the Google Cloud Console
   sa_json = f.read()  # local
if not sa_json:
    raise RuntimeError("The OAuth token was not found. Make sure that the GOOGLE_TOKEN_B64 EV / service_account.json is set.")

creds_info = json.loads(sa_json)

creds = Credentials.from_service_account_info(
    creds_info,
    scopes=["https://www.googleapis.com/auth/drive"]
)
drive_service = build("drive", "v3", credentials=creds)

MY_FOLDER_ID = folder["5 news_lists"] # 5 new lists

In [ ]:

# API_KEY = os.environ.get("PERPLEXITY_API_KEY")  # for workflow
# API_KEY = userdata.get('perplexity_api_key')   # for local

#API_KEY = os.environ.get("DEEPSEEK_API_KEY")  # for workflow
#API_KEY = userdata.get('deepseek_api_key')   # for local


# for local run – from text file (cursor)
import os
from pathlib import Path
#API_KEY_FILE = Path("keys") / "perplexity_api_key.txt"
API_KEY_FILE = Path("keys") / "deepseek_api_key.txt"
#API_KEY_FILE = Path("keys") / "openrouter_api_key.txt"
with open(API_KEY_FILE, "r", encoding="utf-8") as f:
    API_KEY = f.read().strip()

if not API_KEY:
    raise ValueError("There is no API key (check the file or environment variable)!")




# Setting the endpoint and initial messages

#url = "https://api.perplexity.ai/chat/completions"
url = "https://api.deepseek.com/v1/chat/completions"
#url = "https://openrouter.ai/api/v1/chat/completions"


# Setting up models (to test and compare them)

model_lists = "deepseek-chat" # direct deepseek
model_bullets = "deepseek-chat" # direct deepseek
#model_lists = "deepseek/deepseek-chat-v3-0324" # openrouter
#model_bullets = "deepseek/deepseek-chat-v3-0324" #openrouter
# model_bullets = "qwen/qwen-2.5-72b-instruct" #openrouter


headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}



# gemini api key
#API_KEY = os.environ.get("GEMINI_API_KEY") # строка для запуска через workflow
#API_KEY = userdata.get('gemini_api_key') # строка для локального запуска
#genai.configure(api_key=API_KEY)
#model_obj = genai.GenerativeModel(
#    model_name="gemini-2.5-pro",
#    generation_config={
#        "response_mime_type": "application/json",  # ← важно!
#    }
#)

In [131]:
### TG Schedule bot

# Read secrets (set TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID in env or Colab secrets)
#TELEGRAM_BOT_TOKEN = userdata.get("TELEGRAM_BOT_TOKEN")   # set this in env / Colab secrets
#TELEGRAM_CHAT_ID  = userdata.get("TELEGRAM_CHAT_ID")     # set this in env / Colab secrets

if USE_SANDBOX:
    TELEGRAM_BOT_TOKEN = Path("keys") / "TELEGRAM_BOT_TOKEN.txt" #local cursor
    with open(TELEGRAM_BOT_TOKEN, "r", encoding="utf-8") as f:
        TELEGRAM_BOT_TOKEN = f.read().strip()

    TELEGRAM_CHAT_ID = Path("keys") / "TELEGRAM_CHAT_ID.txt" #local cursor
    with open(TELEGRAM_CHAT_ID, "r", encoding="utf-8") as f:
        TELEGRAM_CHAT_ID = f.read().strip()

else:
    TELEGRAM_BOT_TOKEN = Path("keys") / "A" / "TELEGRAM_BOT_TOKEN.txt" #local cursor
    with open(TELEGRAM_BOT_TOKEN, "r", encoding="utf-8") as f:
        TELEGRAM_BOT_TOKEN = f.read().strip()

    TELEGRAM_CHAT_ID = Path("keys") / "A" /  "TELEGRAM_CHAT_ID.txt" #local cursor
    with open(TELEGRAM_CHAT_ID, "r", encoding="utf-8") as f:
        TELEGRAM_CHAT_ID = f.read().strip()




if not TELEGRAM_BOT_TOKEN or not TELEGRAM_CHAT_ID:
    raise RuntimeError("Missing TELEGRAM_BOT_TOKEN or TELEGRAM_CHAT_ID. "
                       "Set them as environment variables or Colab secrets before running.")

def send_telegram_message(text: str,
                          parse_mode: str = "HTML",
                          timeout: int = 10,
                          escape_html: bool = False) -> dict:
    """
    Send a message using the Telegram Bot API.
    Returns the parsed JSON response on success, raises on failure.
    """
    if escape_html:
        text = html.escape(text)

    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    payload = {
        "chat_id": TELEGRAM_CHAT_ID,
        "text": text,
    }
    if parse_mode:
        payload["parse_mode"] = parse_mode

    try:
        # use json=payload so requests sends proper JSON (Telegram accepts both form and JSON)
        resp = requests.post(url, json=payload, timeout=timeout)
        resp.raise_for_status()  # raises on HTTP 4xx/5xx
    except requests.RequestException as e:
        # You could log resp.text here if you keep resp in scope; but be careful not to leak secrets.
        raise RuntimeError(f"Failed to send Telegram message: {e}") from e

    data = resp.json()
    if not data.get("ok"):
        # Telegram sometimes returns 200 OK with {"ok": false, ...}
        raise RuntimeError(f"Telegram API returned an error: {data}")

    return data

def telegram_lists():
    link_news = f"https://drive.google.com/drive/folders/{folder['5 news_lists']}"
    text = f'Новостная записка обновлена. См. отчёты по <a href="{link_news}">ссылке</a>'
    send_telegram_message(text)

def telegram_bullets():
    link_bullets = f"https://drive.google.com/drive/folders/{folder['8 news_final']}"
    text = f'Готовы буллиты к новостной записке. См. отчёты по <a href="{link_bullets}">ссылке</a>'
    send_telegram_message(text)

In [ ]:
### Functions for google drive

def find_file_in_drive(file_name: str, folder_id = folder["5 news_lists"]) -> str: # 5 new lists 
    """Look up a Drive file by exact name inside a folder.

    Args:
        file_name: Exact file name (e.g. ``world.json``).
        folder_id: Google Drive folder ID to search in; defaults to ``5 news_lists``.

    Returns:
        File ID if a matching file exists, otherwise ``None``.
    """  
    try:
        resp = drive_service.files().list(
            q=(
                f"name = '{file_name}' "
                f"and '{folder_id}' in parents "
                f"and trashed = false"
            ),
            spaces="drive",
            fields="files(id, name)",
            pageSize=1
        ).execute()
    except HttpError as e:
        raise RuntimeError(f"Error accessing Drive API: {e}")

    items = resp.get("files", [])
    if items:
        return items[0]["id"]

    return None


def download_text_file(fid: str) -> str:
    """Download a Drive file by ID and return its contents as UTF-8 text.

    Args:
        fid: Google Drive file ID (from ``find_file_in_drive`` or similar).

    Returns:
        File body decoded as a string.
    """
    request = drive_service.files().get_media(fileId=fid)
    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done:
        status, done = downloader.next_chunk()
    return fh.getvalue().decode("utf-8")

def save_to_drive(file_name: str, data, my_folder=MY_FOLDER_ID, file_format: str = "json"):
    """Save a file to Google Drive. Supported formats: 'json' (default) and 'txt'.

    Args:
        file_name: File name.
        data: Data to write (dict or str).
        my_folder: Folder ID in Google Drive.
        file_format: File format: 'json' or 'txt'.
    """
    if file_format not in ("json", "txt"):
        raise ValueError("file_format должен быть 'json' или 'txt'")

    if file_format == "txt":
        content_bytes = data.encode("utf-8") if isinstance(data, str) else str(data).encode("utf-8")
        mime_type = "text/plain"
    else:
        json_str = json.dumps(data, ensure_ascii=False, indent=2)
        content_bytes = json_str.encode("utf-8")
        mime_type = "application/json"

    # Check if file already exists
    existing_file_id = None
    try:
        resp = drive_service.files().list(
            q=f"name = '{file_name}' and '{my_folder}' in parents and trashed = false",
            spaces="drive",
            fields="files(id, name)",
            pageSize=1
        ).execute()
        items = resp.get("files", [])
        if items:
            existing_file_id = items[0]["id"]
    except Exception as e:
        print("Warning: can't check if the file already exists:", e)

    fh = io.BytesIO(content_bytes)
    media = MediaIoBaseUpload(fh, mimetype=mime_type, resumable=False)

    if existing_file_id:
        try:
            # Try to update
            updated = drive_service.files().update(
                fileId=existing_file_id,
                media_body=media
            ).execute()
            print(f"File '{file_name}' updated (ID={updated['id']}).")
            return updated
        except HttpError as e:
            if e.resp.status == 403 and "storageQuotaExceeded" in str(e):
                print(f"⚠️ Quota error on update — deleting and recreating file '{file_name}'...")
                try:
                    drive_service.files().delete(fileId=existing_file_id).execute()
                    existing_file_id = None  # перейти к созданию
                except Exception as del_err:
                    print(f"Error deleting file '{file_name}': {del_err}")
                    raise
            else:
                print(f"Error updating file '{file_name}': {e}")
                raise

    # Create a new file
    file_metadata = {
        "name": file_name,
        "parents": [my_folder],
        "mimeType": mime_type
    }
    try:
        created = drive_service.files().create(
            body=file_metadata,
            media_body=media,
            fields="id, webViewLink"
        ).execute()
        print(f"New file created: '{file_name}', (ID={created['id']}).")
        return created
    except Exception as e:
        print(f"Error creating new file '{file_name}': {e}")
        raise


In [ ]:
PROXY = os.environ.get('PROXY')
proxies = {'https':
        PROXY
        }



PROXY_FILE = Path("keys") / "proxy.txt"
with open(PROXY_FILE, "r", encoding="utf-8") as f:
    PROXY = f.read().strip()
    proxies = {'https':
        PROXY
        }

if not PROXY:
    raise ValueError("There is no proxy (check the file or environment variable)!")


In [ ]:
### Functions for scrapping

## Defining and formatting dates
def get_last_dates(n_days=6, end_date=None):
    if end_date is None:
        end_date = date.today()
    return [end_date - timedelta(days=offset) for offset in range(n_days, -1, -1)]

def format_dates(dates_list, fmt="%Y-%m-%d"):
    return [d.strftime(fmt) for d in dates_list]

## Getting web page soup
def get_page_soup(url, headers=HEADERS, timeout=30):
    resp = requests.get(url, headers=headers, timeout=timeout)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")

## Getting web page soup using session to avoid 401
def get_proxy_page_soup(url, headers=HEADERS, proxies=proxies, timeout=30):
    # We use the session to work with cookies and headers 
    session = requests.Session()

    # First, make a GET on the main page to get the cookie and possibly tokens
    resp = session.get(url, headers=headers,
                        proxies=proxies
                        )
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")



In [84]:
## Scrapers: Kommersant, Vedomosti, RBC, Agroinvestor, RG.ru, RIA, Autostat

# Kommersant scraper
def fetch_kom(rubrics, dates, output_file,
                   base_url_template="https://www.kommersant.ru/archive/rubric/{rubric}/day/{date}"):
    all_items = []
    seen_urls = set()

    for rubric in rubrics:
        for dt in dates:
            url = base_url_template.format(rubric=rubric, date=dt)
            print(f"Fetching Kommersant: {url}")
            try:
                soup = get_page_soup(url)
                scripts = soup.find_all("script", type="application/ld+json")

                for script in scripts:
                    raw = script.string
                    if not raw:
                        continue
                    try:
                        data = json.loads(raw)
                    except json.JSONDecodeError:
                        continue

                    for entry in data.get("itemListElement", []):
                        title = entry.get("name") or entry.get("headline")
                        link = entry.get("url")
                        if title and link and link not in seen_urls:
                            seen_urls.add(link)
                            all_items.append({"title": title, "url": link, "published_date": dt})
            except Exception as e:
                print(f"[ERROR] {e} when fetching {url}")

    save_to_drive(output_file, all_items, folder["1 news_jsons"]) # 1 news_jsons
    print(f"Saved Kommersant data to {output_file}")





In [14]:
# Vedomosti scraper
def fetch_ved(dates, output_file,
              base_url_template="https://www.vedomosti.ru/newspaper/{date}"):
    all_news = []
    for dt in dates:
        url = base_url_template.format(date=dt)
        print(f"Fetching Vedomosti: {url}")
        try:
            soup = get_page_soup(url)
            for item in soup.select("li.waterfall__item"):
                a = item.select_one("a.waterfall__item-title")
                if not a:
                    continue
                title = a.get_text(strip=True)
                href = a.get("href", "")
                full_url = href if href.startswith("http") else f"https://www.vedomosti.ru{href}"
                pub = dt.replace("/", "-") if isinstance(dt, str) else None
                all_news.append({"title": title, "url": full_url, "published_date": pub})
        except Exception as e:
            all_news.append({"error": str(e)})

    save_to_drive(output_file, all_news, folder["1 news_jsons"]) # 1 news_jsons
    print(f"Saved Vedomosti data to {output_file}")


In [15]:
# RBC scraper

def fetch_rbc(rubrics, dates, output_file,
              base_url_template="https://www.rbc.ru/{rubric}/?utm_source=topline"):

    ru_months = {
        'января': 1, 'февраля': 2, 'марта': 3, 'апреля': 4,
        'мая': 5, 'июня': 6, 'июля': 7, 'августа': 8,
        'сентября': 9, 'октября': 10, 'ноября': 11, 'декабря': 12
    }
    today = date.today()
    collected = []

    for rubric in rubrics:
        page_url = base_url_template.format(rubric=rubric)
        print(f"Fetching RBC, {rubric}: {page_url}")
        soup = get_proxy_page_soup(page_url)

        anchors = soup.find_all("a", class_="news-feed__item")

        for idx, a in enumerate(anchors, start=1):
            # внутри anchor ищем span, у которого class содержит "news-feed__item__title"
            title_span = a.find(
                "span",
                class_=lambda c: c and "news-feed__item__title" in c
            )
            if not title_span:
                continue

            # Для даты: ищем span, у которого class содержит "news-feed__item__time"
            # или, если нет, "news-feed__item__date"
            date_span = a.find(
                "span",
                class_=lambda c: c and "news-feed__item__time" in c
            )
            if not date_span:
                date_span = a.find(
                    "span",
                    class_=lambda c: c and "news-feed__item__date" in c
                )
            if not date_span:
                continue

            title = title_span.get_text(strip=True)
            href = a.get("href", "").strip()
            if not href:
                continue

            full_url = href if href.startswith("http") else urljoin(page_url, href)

            # raw_date может быть вида "28 мая 17:52" или просто "17:52"
            raw_date = date_span.get_text(strip=True).replace("\xa0", " ").replace(",", "").strip()
            parts = raw_date.split()

            news_date = None
            if any(month in parts for month in ru_months):
                # формат ["28","мая","17:52"] или ["28","мая","2025","17:52"]
                try:
                    day = int(parts[0])
                except ValueError:
                    continue
                month_name = parts[1].lower()
                if month_name not in ru_months:
                    continue
                month = ru_months[month_name]
                year = today.year
                # если в parts[2] четвёрка цифр, считаем, что это год
                if len(parts) >= 3 and parts[2].isdigit() and len(parts[2]) == 4:
                    year = int(parts[2])
                try:
                    candidate = datetime.date(year, month, day)
                except ValueError:
                    continue
                # если эта дата уже в будущем, значит, год был прошлый
                if candidate > today:
                    candidate = datetime.date(year - 1, month, day)
                news_date = candidate
            else:
                # если нет названия месяца, значит raw_date = "HH:MM" сегодняшняя дата
                news_date = today

            if news_date not in dates:
                continue

            collected.append({
                "title": title,
                "url": full_url,
                "published_date": news_date.isoformat(),
            })

    # убираем дубликаты по URL
    unique = []
    seen = set()
    for item in collected:
        if item["url"] not in seen:
            seen.add(item["url"])
            unique.append(item)

    save_to_drive(output_file, unique, folder["1 news_jsons"]) # 1 news_jsons
    print(f"Saved RBC data to {output_file}")



In [16]:
# Agroinvestor scraper

def fetch_agro(dates, output_file,
               base_url="https://www.agroinvestor.ru/news/"):
    base_domain = "https://www.agroinvestor.ru"
    ru_months = {
        "января": 1, "февраля": 2, "марта": 3, "апреля": 4,
        "мая": 5, "июня": 6, "июля": 7, "августа": 8,
        "сентября": 9, "октября": 10, "ноября": 11, "декабря": 12,
    }

    def parse_once() -> list:
        soup = get_page_soup(base_url)
        if soup is None:
            print("❌ Failed to retrieve or parse the page.")
            return []

        news_list = []
        seen_urls = set()

        for block in soup.select("div.news__item-info"):
            a = block.find("a", class_="news__item-desc")
            if not a:
                continue
            href = a.get("href", "").strip()
            if not href:
                continue
            full_url = href if href.startswith("http") else base_domain + href
            if full_url in seen_urls:
                continue

            h3 = a.find("h3")
            if not h3:
                continue
            title = h3.get_text(strip=True)
            if not title:
                continue

            time_tag = block.find("time")
            if not time_tag:
                continue
            date_text = time_tag.get_text(strip=True).replace("\xa0", " ")
            parts = date_text.split()
            if len(parts) != 3:
                continue
            day_str, month_str, year_str = parts
            try:
                day = int(day_str)
                year = int(year_str)
            except ValueError:
                continue
            month_str = month_str.lower()
            if month_str not in ru_months:
                continue
            month = ru_months[month_str]
            try:
                news_date = date(year, month, day)
            except ValueError:
                continue

            if news_date not in dates:
                continue

            seen_urls.add(full_url)
            news_list.append({
                "title": title,
                "url": full_url,
                "published_date": news_date.isoformat(),
            })

        return news_list

    # первый проход
    news_list = parse_once()

    # если ничего не собрали — один ретрай
    if not news_list:
        print("⚠️ Agroinvestor: empty result, retrying once...")
        news_list = parse_once()

    save_to_drive(output_file, news_list, folder["1 news_jsons"]) # 1 news_jsons
    print(f"Saved {len(news_list)} Agroinvestor items to {output_file}")


In [17]:
#! pip install ipdb
#import ipdb
#%pdb on

In [20]:
# RG.ru scraper

def fetch_rg(rubrics, dates, output_file,
             base_url_template="https://rg.ru/tema/ekonomika/{rubric}"):
    all_news = []
    for rubric in rubrics:
        url = base_url_template.format(rubric=rubric)
        print(f"Fetching RG, {rubric}: {url}")
        soup = get_proxy_page_soup(url)
        for title_span in soup.find_all("span", class_="ItemOfListStandard_title__Ajjlf"):
            parent_a = title_span.find_parent("a")
            if not parent_a:
                continue
            href = parent_a.get("href", "").strip()
            if not href:
                continue
            full_url = href if href.startswith("http") else f"https://rg.ru{href}"

            date_a = title_span.find_previous("a", class_="ItemOfListStandard_datetime__GstJi")
            if not date_a:
                continue
            date_href = date_a.get("href", "").strip()
            parts = date_href.strip("/").split("/")  # ['2025','05','30',...]
            if len(parts) < 3:
                continue
            try:
                y, m, d = map(int, parts[:3])
                news_date = date(y, m, d)
            except ValueError:
                continue

            if news_date not in dates:
                continue

            all_news.append({
                "title": title_span.get_text(strip=True),
                "url": full_url,
                "published_date": news_date.isoformat(),
            })

    unique = []
    seen = set()
    for item in all_news:
        if item["url"] not in seen:
            seen.add(item["url"])
            unique.append(item)

    save_to_drive(output_file, unique, folder["1 news_jsons"]) # 1 news_jsons
    print(f"Saved RG data to {output_file}")



In [21]:
# RIA scraper

def fetch_ria(dates, output_file, base_url_template="https://ria.ru/economy/"):
    print("Fetching RIA: https://ria.ru/economy/")
    soup = get_page_soup(base_url_template)
    collected = []

    # Each news item has <a itemprop="url" href="..."></a>
    for a in soup.find_all("a", itemprop="url"):
        href = a.get("href", "").strip()
        if not href:
            continue
        full_url = href if href.startswith("http") else f"https://ria.ru{href}"

        # Next meta tag with itemprop="name" holds the title
        name_meta = a.find_next("meta", itemprop="name")
        if not name_meta:
            continue
        title = name_meta.get("content", "").strip()
        if not title:
            continue
        parsed = urlparse(full_url)
        parts = parsed.path.lstrip("/").split("/")
        if not parts or len(parts[0]) != 8 or not parts[0].isdigit():
            continue
        y, m, d = int(parts[0][:4]), int(parts[0][4:6]), int(parts[0][6:8])
        try:
            news_date = date(y, m, d)
        except ValueError:
            continue

        if news_date in dates:
            collected.append({
                "title": title,
                "url": full_url,
                "published_date": news_date.isoformat(),
            })

    unique = []
    seen = set()
    for item in collected:
        if item["url"] not in seen:
            seen.add(item["url"])
            unique.append(item)

    save_to_drive(output_file, unique, folder["1 news_jsons"]) # 1 news_jsons

    print(f"Saved RIA data to {output_file}")




In [22]:
# Autostat scraper

def fetch_autostat(dates, output_file,
                   rubrics=[21, 8, 13, 70, 71],
                   base_url_template="https://m.autostat.ru/news/themes-{rubric}/"):

    if dates is None:
        raise ValueError("Argument 'dates' must be provided as a list of datetime.date objects.")

    all_collected = []
    seen_urls = set()

    ru_months = {
        'января': 1, 'февраля': 2, 'марта': 3, 'апреля': 4,
        'мая': 5, 'июня': 6, 'июля': 7, 'августа': 8,
        'сентября': 9, 'октября': 10, 'ноября': 11, 'декабря': 12
    }
    today = date.today()
    yesterday = today - timedelta(days=1)

    for rubric in rubrics:
        url = base_url_template.format(rubric=rubric)
        print(f"Fetching Autostat, {rubric}: {url}")
        soup = get_page_soup(url)
        if not soup:
            print(f"  (!) Failed to retrieve or parse page for rubric {rubric}")
            continue

        titles = soup.find_all("p", class_="Block-title")
        if not titles:
            print(f"    (!) No <p class='Block-title'> elements found on {url}")
            continue

        for title_p in titles:
            title = title_p.get_text(strip=True)
            if not title:
                continue

            link_a = title_p.find_parent("a", class_="Block-link")
            if not link_a:
                continue
            href = link_a.get("href", "").strip()
            if not href:
                continue
            full_url = urljoin("https://www.autostat.ru", href)

            date_p = title_p.find_next("p", class_="Block-date")
            if not date_p:
                continue
            date_text = date_p.get_text(strip=True)  # e.g. "Сегодня, 15:48" or "28 мая, 15:48"
            date_part = date_text.split(",")[0].strip().lower()

            if date_part == "сегодня":
                news_date = today
            elif date_part == "вчера":
                news_date = yesterday
            else:
                parts = date_part.split()
                if len(parts) != 2:
                    continue
                day_str, month_str = parts
                try:
                    day = int(day_str)
                    month = ru_months.get(month_str)
                    if not month:
                        continue
                    news_date = date(today.year, month, day)
                    if news_date > today:
                        news_date = date(today.year - 1, month, day)
                except Exception:
                    continue

            if news_date in dates and full_url not in seen_urls:
                all_collected.append({
                    "title": title,
                    "url": full_url,
                    "published_date": news_date.isoformat(),
                })
                seen_urls.add(full_url)

    save_to_drive(output_file, all_collected, folder["1 news_jsons"]) # 1 news_jsons

    print(f"Saved Autostat data to {output_file}")

In [23]:
#with open('agro.json', encoding='utf-8') as f:
#    data = json.load(f)
#print(json.dumps(data, ensure_ascii=False, indent=2))

In [143]:
# Parameters
days_before = 1 # today + n days before
dates = get_last_dates(days_before) # today + n days before
dates_kom = format_dates(dates, fmt="%Y-%m-%d")
dates_ved = format_dates(dates, fmt="%Y/%m/%d")


rubrics_kom_econ = [3, 4, 40] # 3 - экономика, 4 - бизнеc,  40 - финансы (темы рубрик? для цен топливо в 4 https://www.kommersant.ru/theme/2913 )
rubrics_kom_world = [5] # 5 - мир
rubrics_kom_markets = [41] # 41 - потребительский рынок
rubrics_rbc = ["economics", "business", "finances"]
rubrics_rg = ["politekonom", "industria", "business", "finansy", "kazna", "rabota", "pensii", "vnesh", "apk", "tovary", "turizm"]
rubrics_auto = [21, 8, 13, 70, 71]

In [144]:
dates

[datetime.date(2026, 5, 14), datetime.date(2026, 5, 15)]

In [ ]:
#file creation may not work, there must be an empty blank file

# Fetching
fetch_kom(rubrics_kom_econ, dates_kom, "kom_econ.json")


Fetching Kommersant: https://www.kommersant.ru/archive/rubric/3/day/2026-05-09
Fetching Kommersant: https://www.kommersant.ru/archive/rubric/3/day/2026-05-10
Fetching Kommersant: https://www.kommersant.ru/archive/rubric/3/day/2026-05-11
Fetching Kommersant: https://www.kommersant.ru/archive/rubric/3/day/2026-05-12
Fetching Kommersant: https://www.kommersant.ru/archive/rubric/3/day/2026-05-13
Fetching Kommersant: https://www.kommersant.ru/archive/rubric/3/day/2026-05-14
Fetching Kommersant: https://www.kommersant.ru/archive/rubric/4/day/2026-05-09
Fetching Kommersant: https://www.kommersant.ru/archive/rubric/4/day/2026-05-10
Fetching Kommersant: https://www.kommersant.ru/archive/rubric/4/day/2026-05-11
Fetching Kommersant: https://www.kommersant.ru/archive/rubric/4/day/2026-05-12
Fetching Kommersant: https://www.kommersant.ru/archive/rubric/4/day/2026-05-13
Fetching Kommersant: https://www.kommersant.ru/archive/rubric/4/day/2026-05-14
Fetching Kommersant: https://www.kommersant.ru/archi

In [27]:
fetch_kom(rubrics_kom_world, dates_kom, "kom_world.json")

Fetching Kommersant: https://www.kommersant.ru/archive/rubric/5/day/2026-05-09
Fetching Kommersant: https://www.kommersant.ru/archive/rubric/5/day/2026-05-10
Fetching Kommersant: https://www.kommersant.ru/archive/rubric/5/day/2026-05-11
Fetching Kommersant: https://www.kommersant.ru/archive/rubric/5/day/2026-05-12
Fetching Kommersant: https://www.kommersant.ru/archive/rubric/5/day/2026-05-13
Fetching Kommersant: https://www.kommersant.ru/archive/rubric/5/day/2026-05-14
File 'kom_world.json' updated (ID=1N5hrFb0UjvLf_aoiQ2DdaXw06c3t83Ct).
Saved Kommersant data to kom_world.json


In [28]:
fetch_kom(rubrics_kom_markets, dates_kom, "kom_markets.json")

Fetching Kommersant: https://www.kommersant.ru/archive/rubric/41/day/2026-05-09
Fetching Kommersant: https://www.kommersant.ru/archive/rubric/41/day/2026-05-10
Fetching Kommersant: https://www.kommersant.ru/archive/rubric/41/day/2026-05-11
Fetching Kommersant: https://www.kommersant.ru/archive/rubric/41/day/2026-05-12
Fetching Kommersant: https://www.kommersant.ru/archive/rubric/41/day/2026-05-13
Fetching Kommersant: https://www.kommersant.ru/archive/rubric/41/day/2026-05-14
File 'kom_markets.json' updated (ID=1TPyYHeQmxwCdzfcRpz_sGUXuhm7GkQpx).
Saved Kommersant data to kom_markets.json


In [29]:
fetch_ved(dates_ved, "ved.json")

Fetching Vedomosti: https://www.vedomosti.ru/newspaper/2026/05/09
Fetching Vedomosti: https://www.vedomosti.ru/newspaper/2026/05/10
Fetching Vedomosti: https://www.vedomosti.ru/newspaper/2026/05/11
Fetching Vedomosti: https://www.vedomosti.ru/newspaper/2026/05/12
Fetching Vedomosti: https://www.vedomosti.ru/newspaper/2026/05/13
Fetching Vedomosti: https://www.vedomosti.ru/newspaper/2026/05/14
File 'ved.json' updated (ID=1EQx9Nz8wiFuCCuVTC37h9SqPXIrpyqAk).
Saved Vedomosti data to ved.json


In [30]:
fetch_rbc(rubrics_rbc, dates, "rbc.json")

Fetching RBC, economics: https://www.rbc.ru/economics/?utm_source=topline
Fetching RBC, business: https://www.rbc.ru/business/?utm_source=topline
Fetching RBC, finances: https://www.rbc.ru/finances/?utm_source=topline
File 'rbc.json' updated (ID=1Fdmk78BujFbqUtn5MWj2gpZEbEo_YGGQ).
Saved RBC data to rbc.json


In [32]:
#fetch_rg(rubrics_rg, dates, "rg.json")


Fetching RG, politekonom: https://rg.ru/tema/ekonomika/politekonom


HTTPError: 401 Client Error: Unauthorized for url: https://rg.ru/tema/ekonomika/politekonom

In [31]:
fetch_ria(dates, "ria.json")

Fetching RIA: https://ria.ru/economy/
File 'ria.json' updated (ID=1Kie4tdZL8ZufCsXb-NFV8ETTp2M2lxQy).
Saved RIA data to ria.json


In [32]:
fetch_autostat(dates, "autostat.json", rubrics_auto)

Fetching Autostat, 21: https://m.autostat.ru/news/themes-21/
Fetching Autostat, 8: https://m.autostat.ru/news/themes-8/
Fetching Autostat, 13: https://m.autostat.ru/news/themes-13/
Fetching Autostat, 70: https://m.autostat.ru/news/themes-70/
Fetching Autostat, 71: https://m.autostat.ru/news/themes-71/
File 'autostat.json' updated (ID=13W2RWphViuK9LPPo-p2fSgLFgZ0Lq17a).
Saved Autostat data to autostat.json


In [34]:
try:
    fetch_rg(rubrics_rg, dates, "rg.json")
except Exception as e:

    pass

Fetching RG, politekonom: https://rg.ru/tema/ekonomika/politekonom


In [35]:
try:
    fetch_agro(dates, "agro.json")
except Exception as e:

    pass

File 'agro.json' updated (ID=1vtJd3OuAYlG24s9XZrafwx0ItmHsHF-k).
Saved 12 Agroinvestor items to agro.json


In [64]:
# Kommersant, Vedomosti, RBC, Agroinvestor, RG.ru, RIA, Autostat
section_to_files = {
    "world": [
        "kom_world.json",
        "kom_econ.json",
        "ved.json",
        "rbc.json",
        "agro.json",
        #"rg.json",
        "ria.json"
    ],
    "rus": [
        "kom_econ.json",
        "ved.json",
        "rbc.json",
        "agro.json",
        #"rg.json",
        "ria.json"
    ],
    "prices": [
        "kom_markets.json",
        "kom_econ.json",
        "ved.json",
        "rbc.json",
        "agro.json",
        #"rg.json",
        "ria.json",
        "autostat.json"
    ]
}

In [37]:
#drive.mount('/content/drive')

In [ ]:
# Prompts

## download prompts from drive

### news lists
file_id = find_file_in_drive("lists_world.txt", folder["0_prompts"]) # 0 prompts
try:
    lists_world = download_text_file(file_id)
except Exception as e:
    print("Error when downloading a file:", e)
    lists_world = ""

file_id = find_file_in_drive("lists_rus.txt", folder["0_prompts"])
try:
    lists_rus = download_text_file(file_id)
except Exception as e:
    print("Error when downloading a file:", e)
    lists_rus = ""

file_id = find_file_in_drive("lists_prices.txt", folder["0_prompts"])
try:
    lists_prices = download_text_file(file_id)
except Exception as e:
    print("Error when downloading a file:", e)
    lists_prices = ""

lists_prompts = {
        "world": lists_world,
        "rus": lists_rus,
        "prices": lists_prices
}



In [88]:
lists_prices

'### Роль: Ассистент-макроэкономист — эксперт по российской экономике и ценовой динамике. Отбирает важные и уникальные новости, исключая локальные, с участием Банка России и не относящиеся к динамике цен в России. Строго фильтрует по заданным критериям, работает только с приложенным списком. Форматирует результат в JSON с не более, чем 50 новостями без лишних комментариев.\r\n\r\n### Цель: Отобрать НЕ БОЛЕЕ 50 релевантных новостей для раздела «Новости, релевантные для динамики цен» строго из предоставленного списка, при этом сохраняя достаточную насыщенность раздела.\r\n\r\n### Контекст:\r\n## Общее:\r\nТы подготавливаешь блок аналитической записки. Кандидаты-новости предоставлены в единственном файле. Твоё задание — выбрать только те, что соответствуют всем критериям ниже. Особое внимание уделяй новостям из файлов agro.json и autostat.json — они с высокой вероятностью должны остаться в итоговом списке.\r\n\r\n## Что нельзя делать:\r\n- Запрещено использовать любые источники, кроме при

In [ ]:
### prioritise
file_id = find_file_in_drive("prioritise_world.txt", folder["0_prompts"])
try:
    prioritise_world = download_text_file(file_id)
except Exception as e:
    print("Error when downloading a file:", e)
    prioritise_world = ""

file_id = find_file_in_drive("prioritise_rus.txt", folder["0_prompts"])
try:
    prioritise_rus = download_text_file(file_id)
except Exception as e:
    print("Error when downloading a file:", e)
    prioritise_rus = ""

file_id = find_file_in_drive("prioritise_prices.txt", folder["0_prompts"])
try:
    prioritise_prices = download_text_file(file_id)
except Exception as e:
    print("Error when downloading a file:", e)
    prioritise_prices = ""

prioritise_prompts = {
        "world": prioritise_world,
        "rus": prioritise_rus,
        "prices": prioritise_prices
}



In [90]:
prioritise_prices

'Ты — эксперт по анализу ценовой динамики в России. Твоя задача: строго отфильтровать предоставленные новости и выбрать максимум 40 самых релевантных для раздела «Динамика цен в России», присвоив каждой оценку от 0 до 10.\r\n\r\n## ВАЖНО: Правила использования информации\r\n- Ты можешь анализировать ТОЛЬКО тексты новостей, предоставленные в приложенном JSON\r\n- Ты НЕ можешь искать новые ссылки или добавлять новости извне\r\n- Ты можешь использовать свои знания до 2026 года ТОЛЬКО для контекстного понимания\r\n- Все решения должны основываться на предоставленных текстах\r\n\r\n## Приоритетность тем для ЦБ РФ (в порядке убывания важности)\r\n1. **Регулирование**: введение новых налогов, сборов, пошлин, акцизов, квот на любые товары и услуги\r\n2. **Топливо**: ситуация на российском внутреннем рынке бензина и дизеля\r\n3. **Авто**: ситуация на российском автомобильном рынке\r\n4. **Маркетплейсы**: регулирование маркетплейсов\r\n5. **Ближний Восток**: последствия конфликта на Ближнем Вост

In [ ]:
### design

file_id = find_file_in_drive("design.txt", folder["0_prompts"])
try:
    prompt_design = download_text_file(file_id)
except Exception as e:
    print("Error when downloading a file:", e)
    prompt_design = ""



In [92]:
prompt_design

'Ты создан, чтобы помогать макроэкономисту в подготовке еженедельной аналитической записки о наиболее важных новостях, произошедших за указанный период и важных для российской экономики.\r\n\r\nТебе нужно СТРОГО НА ОСНОВЕ СПИСКА, КОТОРЫЙ Я ПРИЛОЖУ, оформить нумерованный список новостей в соответствии с "Требованиями к оформлению нумерованного списка":\r\nB1. В заголовке новости не упоминается источник данных, только суть произошедшего.\r\nB2. Заголовок заканчивается меткой о дне недели из столбца published_date (например: "(published_date: 2026-05-11)")\r\nB3. Сразу под заголовком прямая URL-ссылка на конкретную новость.\r\nB4. Ссылка на новость должна соответствовать новости в заголовке.\r\nB5. Заголовок новости на русском языке.\r\n\r\n## Твое текущее задание\r\nПожалуйста, оформи прилагаемый нумерованный список так: новость с указанием дня недели, ниже ее URL, прикладываю пример оформления. ОЧЕНЬ ВАЖНО: В ОТВЕТ НЕ ПРИСЫЛАЙ НИЧЕГО КРОМЕ ТЕКСТОВОГО ФАЙЛА.'

In [ ]:
### top
file_id = find_file_in_drive("top_world.txt", folder["0_prompts"])
try:
    top_world = download_text_file(file_id)
except Exception as e:
    print("Error when downloading a file:", e)
    top_world = ""

file_id = find_file_in_drive("top_rus.txt", folder["0_prompts"])
try:
    top_rus = download_text_file(file_id)
except Exception as e:
    print("Error when downloading a file:", e)
    top_rus = ""

file_id = find_file_in_drive("top_prices.txt", folder["0_prompts"])
try:
    top_prices = download_text_file(file_id)
except Exception as e:
    print("Error when downloading a file:", e)
    top_prices = ""

top_prompts = {
        "world": top_world,
        "rus": top_rus,
        "prices": top_prices
}



In [94]:
top_prices

'Ты — макроэкономический аналитик, специализирующийся на ценовой динамике в России. Твоя задача — проанализировать предоставленный список новостей, выделить 4 самых значимых темы и сгруппировать соответствующие статьи.\r\n\r\n## Правила работы\r\n- ✅ Можешь читать содержимое статей по предоставленным ссылкам для точного определения темы\r\n- ✅ Используй ТОЛЬКО предоставленные новости (НЕ ищи информацию в интернете)\r\n- ✅ Темы должны соответствовать приоритетности для ЦБ РФ (см. ниже)\r\n- ❌ НЕ включай новости с упоминанием ЦБ РФ как источника\r\n- ❌ НЕ создавай новые темы вне заданной структуры\r\n\r\n## Приоритетность тем (в порядке убывания важности)\r\n1. **Регулирование**: введение новых налогов, сборов, пошлин, акцизов, квот\r\n2. **Топливо**: ситуация на внутреннем рынке бензина и дизеля\r\n3. **Авто**: ситуация на автомобильном рынке РФ\r\n4. **Маркетплейсы**: регулирование и динамика цен на маркетплейсах\r\n5. **Ближний Восток**: последствия конфликта для цен в России\r\n6. **

In [ ]:
### bullets
file_id = find_file_in_drive("bullets_world.txt", folder["0_prompts"])
try:
    bullets_world = download_text_file(file_id)
except Exception as e:
    print("Error when downloading a file:", e)
    bullets_world = ""

file_id = find_file_in_drive("bullets_rus.txt", folder["0_prompts"])
try:
    bullets_rus = download_text_file(file_id)
except Exception as e:
    print("Error when downloading a file:", e)
    bullets_rus = ""

file_id = find_file_in_drive("bullets_prices.txt", folder["0_prompts"])
try:
    bullets_prices = download_text_file(file_id)
except Exception as e:
    print("Error when downloading a file:", e)
    bullets_prices = ""

bullets_prompts = {
        "world": bullets_world,
        "rus": bullets_rus,
        "prices": bullets_prices
}

example = 'Пример верного оформления:\r\n1.\tРосстат зафиксировал стабилизацию выпуска базовых отраслей (day: 3) \r\nhttps://www.kommersant.ru/doc/7329366 \r\n2.\tСтроители просят смягчить правила распоряжения авансами (day: 1)\r\nhttps://www.rbc.ru/newspaper/2024/11/25/673f6abf9a7947de58a24847 \r\n3.\tВ Ульяновске открылся новый завод грузовиков Соллерс (day: 0) \r\nhttps://tass.ru/ekonomika/22497349 \r\n 4.\t Добыча газа за 9 месяцев выросла на 8% г/г в основном за счет Газпрома (day: 3) \r\nhttps://www.interfax.ru/business/994801 \r\n'


In [ ]:
def extract_json(text: str):
    """Extract a valid JSON object or array from an arbitrary text string.

    The function tries several strategies, in order of reliability:

    1. Look for a fenced code block (``` ... ```) and parse its content.
    2. Find the longest balanced ``[...]`` or ``{...}`` fragment that
       parses as valid JSON.
    3. If the whole text is a JSON-encoded string, unescape it and try
       to parse the result.
    4. Brute-force scan: try every substring that starts with ``[``/``{``
       and ends with ``]``/``}`` (slow fallback for badly formatted output).

    Args:
        text: Raw string that may contain JSON mixed with other text
            (e.g. LLM response with explanations).

    Returns:
        Parsed Python object (``dict`` or ``list``) on success,
        ``None`` if no valid JSON fragment could be recovered.
    """
    if not isinstance(text, str):
        return None
    text = text.strip()
    # Шаг 1: Ищем кодовые блоки (наиболее надёжный способ)
    code_block_match = re.search(r"``````", text, re.IGNORECASE)
    if code_block_match:
        candidate = code_block_match.group(1).strip()
        try:
            return json.loads(candidate)
        except json.JSONDecodeError as e:
            print(f"❌ JSON в кодовом блоке невалиден: {e}")
    # Шаг 2: Ищем самый длинный возможный JSON-массив или объект
    brackets = []
    for i, char in enumerate(text):
        if char in '[{':
            brackets.append((i, char))
        elif char in ']}':
            brackets.append((i, char))
    stack = []
    ranges = []
    for pos, char in brackets:
        if char in '[{':
            stack.append((pos, char))
        elif char == ']' and stack and stack[-1][1] == '[':
            start, _ = stack.pop()
            ranges.append((start, pos))
        elif char == '}' and stack and stack[-1][1] == '{':
            start, _ = stack.pop()
            ranges.append((start, pos))
    ranges.sort(key=lambda x: x[1] - x[0], reverse=True)
    for start, end in ranges:
        candidate = text[start:end+1]
        try:
            result = json.loads(candidate)
            if isinstance(result, (dict, list)) and len(result) >= 0:
                return result
        except json.JSONDecodeError:
            continue
    # Шаг 3: Фолбэк — попытка убрать экранирование
    if text.startswith('"') and text.endswith('"'):
        try:
            unescaped = text[1:-1].encode().decode('unicode_escape')
            if (unescaped.startswith('{') and unescaped.endswith('}')) or \
               (unescaped.startswith('[') and unescaped.endswith(']')):
                return json.loads(unescaped)
        except Exception:
            pass
    # Шаг 4: Крайний фолбэк — перебор подстрок (медленно)
    for start in range(len(text)):
        if text[start] not in '[{':
            continue
        for end in range(len(text), start, -1):
            if text[end-1] not in ']}':
                continue
            fragment = text[start:end]
            if len(fragment) < 3:
                continue
            try:
                return json.loads(fragment)
            except json.JSONDecodeError:
                continue
    return None

def _normalize_url_key(u: str) -> str:
    """Normalize a URL so it can be used as a dictionary lookup key.

    Trims surrounding whitespace and strips a single trailing slash.

    Args:
        u: Raw URL string (or any value).

    Returns:
        Normalized URL, or an empty string for falsy/non-string input.
    """
    if not u or not isinstance(u, str):
        return ""
    u = u.strip()
    if len(u) > 1 and u.endswith("/"):
        u = u.rstrip("/")
    return u


def _published_date_map_from_feed(news_data) -> dict:
    """Build a ``{url: published_date}`` map from a raw news feed JSON.

    Used to restore the original publication date of a news item after the
    LLM has filtered/reformatted the feed (the model may drop the
    ``published_date`` field).

    Args:
        news_data: List of news dicts coming straight from a scraper file
            (each dict typically has ``url`` and ``published_date`` keys).

    Returns:
        Dict mapping normalized URL (see ``_normalize_url_key``) to the
        ``published_date`` string (``YYYY-MM-DD``).
    """
    out = {}
    rows = news_data if isinstance(news_data, list) else []
    for row in rows:
        if not isinstance(row, dict) or "error" in row:
            continue
        u = row.get("url")
        p = row.get("published_date")
        if u and p:
            out[_normalize_url_key(u)] = p
    return out


In [98]:
# def extract_json(text: str):
#     """
#     Пытается найти в строке `text` JSON-список или JSON-объект и вернуть его как Python-структуру.
#     Сначала ищет JSON-массив [...], если не находит — JSON-объект {...}.
#     Если подходящего фрагмента нет или он невалиден — возвращает None.
#     """
#     # 1) Пытаемся найти JSON-массив: ищем первую '[' и последнюю ']'
#     start = text.find('[')
#     end = text.rfind(']')
#     if 0 <= start < end:
#         candidate = text[start:end+1]
#         try:
#             return json.loads(candidate)
#         except json.JSONDecodeError:
#             pass

#     # 2) Если не найден массив, пробуем найти JSON-объект: первую '{' и последнюю '}'
#     start = text.find('{')
#     end = text.rfind('}')
#     if 0 <= start < end:
#         candidate = text[start:end+1]
#         try:
#             return json.loads(candidate)
#         except json.JSONDecodeError:
#             pass

#     # 3) Если ни то, ни другое не получилось — отдадим None
#     return None

In [99]:

# #deepseek trial
# # Please install OpenAI SDK first: `pip3 install openai`
# import os
# from openai import OpenAI

# client = OpenAI(api_key=API_KEY, base_url="https://api.deepseek.com")

# response = client.chat.completions.create(
#     model="deepseek-chat",
#     messages=[
#         {"role": "system", "content": "You are a helpful assistant"},
#         {"role": "user", "content": "Hello"},
#     ],
#     stream=False
# )

# print(response.choices[0].message.content)



In [ ]:
def create_news_lists(section):
    """Build a per-section news list by filtering raw scraper feeds with an LLM.

    For the given section (``"world"``, ``"rus"`` or ``"prices"``) the
    function:

    1. On any weekday except Saturday, loads the previously saved
       ``<section>.json`` from the ``2 4 new_lists_json`` Drive folder so
       results accumulate over the week. On Saturday it starts from an
       empty list.
    2. Iterates over the scraper feeds declared in
       ``section_to_files[section]`` (Kommersant, Vedomosti, RBC,
       Agroinvestor, RIA, Autostat, ...), loads each JSON from Drive and
       sends it to the chat completions API together with the
       section-specific prompt from ``lists_prompts[section]``.
    3. Parses the model response (expected to be a JSON list of
       ``{title, url}``), deduplicates by URL against items already kept,
       and re-attaches the original ``published_date`` from the raw feed
       via ``_published_date_map_from_feed``.
    4. Saves the combined result back to Drive as ``<section>.json``
       in the ``2 4 new_lists_json`` folder.

    Args:
        section: One of ``"world"``, ``"rus"``, ``"prices"`` — determines
            which feeds and which prompt are used.

    Returns:
        None. The function logs progress and writes the result to Drive.
    """
    current_weekday_num = datetime.today().weekday()

    # Если сегодня не суббота — пробуем прочитать уже сохранённый <section>.json
    if current_weekday_num != 5:  # 5 = Saturday
        try:
            existing_id = find_file_in_drive(f"{section}.json", folder["2 4 new_lists_json"]) # 2 4 news lists json 
            existing_text = download_text_file(existing_id)
            try:
                combined_items = json.loads(existing_text)
            except json.JSONDecodeError:
                combined_items = []
        except Exception:
            combined_items = []
    else:
        combined_items = []

    # Обновляем seen_urls и подготавливаем set для сохранённых URL
    seen_urls = {item["url"] for item in combined_items if isinstance(item, dict) and "url" in item}

    # Список файлов и промпт для секции
    json_files = section_to_files[section]
    prompt_list = lists_prompts.get(section, "")

    for json_filename in json_files:
        base_name, ext = os.path.splitext(json_filename)
        if ext.lower() != ".json":
            print(f"Пропускаем '{json_filename}', т.к. не .json-файл.")
            continue

        # Загружаем JSON-файл из Google Drive
        try:
            file_id = find_file_in_drive(json_filename, folder["1 news_jsons"]) # 1 news_jsons
            raw_text = download_text_file(file_id)
        except FileNotFoundError:
            print(f"Файл '{json_filename}' не найден. Пропускаем.")
            continue
        except Exception as e:
            print(f"Ошибка при скачивании '{json_filename}': {e}. Пропускаем.")
            continue

        if not isinstance(raw_text, str) or not raw_text.strip():
            print(f"JSON '{json_filename}' пустой. Пропускаем.")
            continue

        try:
            news_data = json.loads(raw_text)
        except json.JSONDecodeError as e:
            print(f"Ошибка JSON в '{json_filename}': {e}. Пропускаем.")
            continue

        if isinstance(news_data, (list, dict)) and len(news_data) == 0:
            print(f"JSON '{json_filename}' содержит пустую структуру. Пропускаем.")
            continue

        rows_for_dates = (
            news_data
            if isinstance(news_data, list)
            else ([news_data] if isinstance(news_data, dict) else [])
        )
        published_map = _published_date_map_from_feed(rows_for_dates)

        # Формируем prompt для модели
        news_json_string = json.dumps(news_data, ensure_ascii=False, indent=2)
        prompt_parts = [
            str(prompt_list),
            str(news_json_string)
        ]

        # Запрос к model API
        try:
            payload = {
                "model": model_lists,
                "messages": [
                    {"role": "system", "content": "Отвечай строго в формате JSON. Никогда не добавляй в списки новостей источники, найденные в интернете - отбирай новости только из приложенного списка."},
                    {"role": "user", "content": "\n".join(prompt_parts)}
                ],
                "temperature": 0.2,
                "response_format": {"type": "json_object"}
            }
            response = requests.post(url, headers=headers, json=payload)
            response.raise_for_status()
            result = response.json()


        # # Запрос к Perplexity API
        # try:
        #     payload = {
        #         "model": "sonar-pro",
        #         "messages": [
        #             {"role": "system", "content": "Отвечай строго в формате JSON. Никогда не добавляй в списки новостей источники, найденные в интернете - отбирай новости только из приложенного списка."},
        #             {"role": "user", "content": "\n".join(prompt_parts)}
        #         ],
        #         "temperature": 0.2,
        #         "response_mime_type": "application/json",
        #         "disable_search": True
        #     }
        #     response = requests.post(url, headers=headers, json=payload)
        #     response.raise_for_status()
        #     result = response.json()



            # Проверка, что модель вернула контент
            choices = result.get("choices")
            if not choices or not choices[0].get("message", {}).get("content"):
                print(f"Модель не вернула ответ для '{json_filename}'. Пропускаем.")
                continue

            assistant_json_str = choices[0]["message"]["content"]

            # Парсим JSON из строки (ожидается список словарей с ключами 'url' и 'title')
            try:
                items = json.loads(assistant_json_str)
            except json.JSONDecodeError as e:
                print(f"Ответ модели для '{json_filename}' не содержит валидный JSON: {e}")
                continue

            # Приводим к списку, если словарь
            if isinstance(items, dict):
                items = [items]

            if not isinstance(items, list):
                print(f"Ответ модели для '{json_filename}' вернул не список, а {type(items)}. Пропускаем.")
                continue

            # Фильтруем и добавляем новости; published_date из сырого фида по URL
            for entry in items:
                url_val = entry.get("url")
                title_val = entry.get("title")
                if not title_val or not url_val or url_val in seen_urls:
                    continue
                seen_urls.add(url_val)
                pub = published_map.get(_normalize_url_key(url_val)) or entry.get("published_date")
                combined_items.append({
                    "title": title_val,
                    "url": url_val,
                    "published_date": pub,
                })

        except Exception as e:
            print(f"Ошибка при вызове модели для '{json_filename}': {e}. Пропускаем.")
            continue

    if not combined_items:
        print(f"For section '{section}', zero JSONs were successfully processed.")
        return

    # Сохраняем объединённый результат (published_date — дата из фида, если была)
    output_file = f"{section}.json"
    save_to_drive(output_file, combined_items, my_folder=folder["2 4 new_lists_json"])
    print(f"✅ create_news_lists({section}) — успешно обработан и сохранён файл.")

    if not combined_items:
        print(f"For section '{section}', zero JSONs were successfully processed.")
        return

    # Сохраняем объединённый результат
    output_file = f"{section}.json"
    save_to_drive(output_file, combined_items, my_folder=folder["2 4 new_lists_json"])
    print(f"✅ create_news_lists({section}) — успешно обработан и сохранён файл.")


In [101]:
# def create_news_lists(section):

#     # Если сегодня не суббота, пробуем прочитать существующий файл <section>.json
#     if datetime.today().weekday() != 5:  # 5 = Saturday
#         try:
#             existing_id = find_file_in_drive(f"{section}.json", folder["2 4 new_lists_json"])
#             existing_text = download_text_file(existing_id)
#             try:
#                 combined_items = json.loads(existing_text)
#             except json.JSONDecodeError:
#                 combined_items = []
#         except Exception:
#             combined_items = []
#     else:
#         combined_items = []

#     seen_urls = {item["url"] for item in combined_items if isinstance(item, dict) and "url" in item}

#     # Достаём список JSON-файлов и prompt_list_continue
#     json_files = section_to_files[section]
#     prompt_list_continue = section_to_continue_prompt[section]

#     for json_filename in json_files:
#         base_name, ext = os.path.splitext(json_filename)
#         if ext.lower() != ".json":
#             print(f"Пропускаем '{json_filename}', т.к. не .json-файл.")
#             continue

#         try:
#             file_id = find_file_in_drive(json_filename, folder["1 news_jsons"])
#             raw_text = download_text_file(file_id)
#         except FileNotFoundError:
#             print(f"Файл '{json_filename}' не найден. Пропускаем.")
#             continue
#         except Exception as e:
#             print(f"Ошибка при скачивании '{json_filename}': {e}. Пропускаем.")
#             continue

#         if not isinstance(raw_text, str) or not raw_text.strip():
#             print(f"JSON '{json_filename}' пустой. Пропускаем.")
#             continue

#         try:
#             news_data = json.loads(raw_text)
#         except json.JSONDecodeError as e:
#             print(f"Ошибка JSON в '{json_filename}': {e}. Пропускаем.")
#             continue

#         if isinstance(news_data, (list, dict)) and len(news_data) == 0:
#             print(f"JSON '{json_filename}' содержит пустую структуру. Пропускаем.")
#             continue

#         news_json_string = json.dumps(news_data, ensure_ascii=False, indent=2)

#         raw_parts = [
#             news_json_string,
#             prompt_list_start,
#             prompt_list_continue,
#             prompt_list_finish
#         ]

#         prompt_parts = []
#         for part in raw_parts:
#             if isinstance(part, list):
#                 prompt_parts.append("\n".join(part))
#             else:
#                 prompt_parts.append(str(part))

#         try:
#             response = model_obj.generate_content(prompt_parts)
#         except Exception as e:
#             print(f"Error in model.generate_content for '{json_filename}': {e}.")
#             continue

#         raw_reply = response.text

#         # Попытка вычленить JSON-список (или объект) из текста модели:
#         items = extract_json(raw_reply)
#         if items is None:
#             print(f"Ответ модели для '{json_filename}' не содержит валидный JSON:\n{raw_reply[:200]}… Пропускаем.")
#             continue

#         # Если получили одиночный объект (dict), обернём в список:
#         if isinstance(items, dict):
#             items = [items]

#         if not isinstance(items, list):
#             print(f"Ответ модели для '{json_filename}' вернул не список, а {type(items)}. Пропускаем.")
#             continue

#         # Дальше идёт проверка каждого entry в items: title, url и т.д.
#         for entry in items:
#             url = entry.get("url")
#             title = entry.get("title")
#             if not title or not url or url in seen_urls:
#                 continue
#             seen_urls.add(url)
#             combined_items.append({"title": title, "url": url})

#     if not combined_items:
#         print(f"For section '{section}', zero JSONs were successfully processed.")
#         return

#     # Сохраняем объединённый JSON в файл <section>.json
#     output_file = f"{section}.json"
#     save_to_drive(output_file, combined_items, my_folder = folder["2 4 new_lists_json"])


In [102]:
# Kommersant, Vedomosti, RBC, Agroinvestor, RG.ru, RIA, Autostat
create_news_lists("world")
time.sleep(60)
create_news_lists("rus")
time.sleep(60)
create_news_lists("prices")

JSON 'rbc.json' содержит пустую структуру. Пропускаем.
File 'world.json' updated (ID=1MP9nbf9XX9FehFLquh4KjIwyME7RzXsv).
✅ create_news_lists(world) — успешно обработан и сохранён файл.
File 'world.json' updated (ID=1MP9nbf9XX9FehFLquh4KjIwyME7RzXsv).
✅ create_news_lists(world) — успешно обработан и сохранён файл.
JSON 'rbc.json' содержит пустую структуру. Пропускаем.
File 'rus.json' updated (ID=1rZ9LaJ35ItO1wSCf9YZN1ECkoGY7Kg0n).
✅ create_news_lists(rus) — успешно обработан и сохранён файл.
File 'rus.json' updated (ID=1rZ9LaJ35ItO1wSCf9YZN1ECkoGY7Kg0n).
✅ create_news_lists(rus) — успешно обработан и сохранён файл.
JSON 'rbc.json' содержит пустую структуру. Пропускаем.
File 'prices.json' updated (ID=1FZ_dQmX7zUMYeu0u041hgh3YaX0_c7P6).
✅ create_news_lists(prices) — успешно обработан и сохранён файл.
File 'prices.json' updated (ID=1FZ_dQmX7zUMYeu0u041hgh3YaX0_c7P6).
✅ create_news_lists(prices) — успешно обработан и сохранён файл.


In [ ]:
def prioritise(section):
    """Re-rank the section news list with an LLM and keep the top 40 items.

    Loads ``<section>.json`` from the ``2 4 new_lists_json`` Drive folder,
    feeds it together with the section-specific prompt from
    ``prioritise_prompts[section]`` to the DeepSeek chat API, and expects
    the model to return a JSON array of ``{title, url, published_date, grade}``.

    The model output is:

    * stored as-is in the ``3 news_lists_json_grade`` folder (debug copy
      with grades);
    * sorted by ``grade`` descending and trimmed to 40 items
      (or simply truncated to 40 if grades are missing);
    * stripped to ``{title, url, published_date}`` and written back to
      ``<section>.json`` in the ``2 4 new_lists_json`` folder, replacing
      the previous list.

    Args:
        section: One of ``"world"``, ``"rus"``, ``"prices"``.

    Returns:
        None. Logs progress and writes results to Drive; aborts early on
        any I/O or model error.
    """
    file_name = f"{section}.json"
    folder_id = folder["2 4 new_lists_json"] # 2 4 new_lists_json
    temp_folder_id = folder["3 news_lists_json_grade"] # 3 grade
    combined_items = []
    # Загружаем файл с новостями
    try:
        file_id = find_file_in_drive(file_name, folder_id)
        news_list_raw = download_text_file(file_id)
    except FileNotFoundError:
        print(f"❌ Файл {file_name} не найден в папке {folder_id}.")
        return
    except Exception as e:
        print(f"❌ Ошибка при загрузке файла {file_name}: {e}")
        return
    if not news_list_raw.strip():
        print(f"❌ Файл {file_name} пустой.")
        return
    # Готовим prompt
    prompt_prioritise = prioritise_prompts.get(section, "")
    prompt_text = "\n".join([str(prompt_prioritise), news_list_raw])
    
    try:
        payload = {
            "model": model_bullets,
           "messages": [
               {"role": "system", "content": "Отвечай строго в формате JSON. Никогда не добавляй в списки новостей источники, найденные в интернете - отбирай новости только из приложенного списка."},
                {"role": "user", "content": prompt_text}
             ],
            "temperature": 0.2,
            "response_format": {"type": "json_object"}
        }
    
    # try:
    #     payload = {
    #         "model": "sonar-pro",
    #         "messages": [
    #             {"role": "system", "content": "Отвечай строго в формате JSON. Никогда не добавляй в списки новостей источники, найденные в интернете - отбирай новости только из приложенного списка."},
    #             {"role": "user", "content": prompt_text}
    #         ],
    #         "temperature": 0.2,
    #         "response_mime_type": "application/json",
    #         "disable_search": True
    #     }
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()
        result = response.json()
        
        
        # Проверка ответа
        choices = result.get("choices")
        if not choices or not choices[0].get("message", {}).get("content"):
            print(f"❌ Модель не вернула ответ для '{file_name}'.")
            return
        
        assistant_json_str = choices[0]["message"]["content"]
        # Отладка - вывод ответа модели перед парсингом JSON
        print("DEBUG: Ответ модели (первые 3000 символов):")
        print(assistant_json_str[:3000])
        
        try:
            items = json.loads(assistant_json_str)
        except json.JSONDecodeError as e:
            print(f"❌ Ответ модели для '{file_name}' не содержит валидный JSON: {e}")
            return
        if isinstance(items, dict):
            items = [items]
        if not isinstance(items, list):
            print(f"❌ Ответ модели для '{file_name}' вернул не список, а {type(items)}.")
            return
        
        # Сохраняем полный ответ в отдельную папку
        save_to_drive(file_name, items, temp_folder_id, file_format="json")
        # Обработка с grade
        if all(isinstance(entry, dict) and "grade" in entry for entry in items):
            items_sorted = sorted(items, key=lambda x: x["grade"], reverse=True)
            items_top40 = items_sorted[:40]
            combined_items = [
                {
                    "title": e.get("title"),
                    "url": e.get("url"),
                    "published_date": e.get("published_date"),
                }
                for e in items_top40 if e.get("url")
            ]
        else:
            # Нет grade — берем первые 40 записей с валидным url
            combined_items = [
                {
                    "title": e.get("title"),
                    "url": e.get("url"),
                    "published_date": e.get("published_date"),
                }
                for e in items if e.get("url")
            ][:40]
    except Exception as e:
        print(f"❌ Ошибка при вызове модели для '{file_name}': {e}")
        return
    # Сохраняем итоговый результат в исходную папку
    save_to_drive(file_name, combined_items, folder_id, file_format="json")
    print(f"✅ prioritise({section}) — сохранён корректный JSON.")


In [104]:
prioritise("world")
time.sleep(60)
prioritise("rus")
time.sleep(60)
prioritise("prices")

DEBUG: Ответ модели (первые 3000 символов):
[
  {
    "title": "Инфляция в США в апреле выросла до 3,8% впервые за три года",
    "url": "https://www.kommersant.ru/doc/8654367",
    "published_date": "2026-05-13",
    "grade": 9.8
  },
  {
    "title": "МЭА ухудшило прогноз по мировому спросу на нефть в 2026 году",
    "url": "https://www.kommersant.ru/doc/8654644",
    "published_date": "2026-05-13",
    "grade": 9.5
  },
  {
    "title": "ОПЕК снизила прогноз по росту мирового спроса на нефть в 2026 году",
    "url": "https://www.kommersant.ru/doc/8654854",
    "published_date": "2026-05-13",
    "grade": 9.4
  },
  {
    "title": "Bloomberg сообщило о рекордном истощении мировых запасов нефти",
    "url": "https://www.kommersant.ru/doc/8652932",
    "published_date": "2026-05-10",
    "grade": 9.3
  },
  {
    "title": "Цена нефти Brent превысила $105 после отказа Трампа от мирного плана Ирана",
    "url": "https://www.kommersant.ru/doc/8653520",
    "published_date": "2026-05-12",


In [108]:
def design_wo_llm(section):
    """Render the section news list as a plain numbered text file (no LLM).

    Loads ``<section>.json`` from the ``2 4 new_lists_json`` Drive folder
    and formats every item as::

        N.\\t<title> (published: <published_date>)
        <url>

    Items without ``title`` or ``url`` are skipped. The resulting text is
    saved as ``<section>.txt`` in the ``5 news_lists`` Drive folder. This
    is the cheap fallback used in the daily pipeline; ``design`` is the
    LLM-based variant.

    Args:
        section: One of ``"world"``, ``"rus"``, ``"prices"``.

    Returns:
        None. Writes the formatted ``.txt`` to Drive.
    """
    file_name_json = f"{section}.json"
    try:
        file_id = find_file_in_drive(file_name_json, folder["2 4 new_lists_json"]) # 2 4 new_lists_json
        news_list_raw = download_text_file(file_id)
    except FileNotFoundError:
        print(f"Файл {file_name_json} не найден в папке.")
        return
    except Exception as e:
        print(f"Ошибка при загрузке файла {file_name_json}: {e}")
        return
    # Парсим входной JSON
    try:
        news_items = json.loads(news_list_raw)
    except json.JSONDecodeError as e:
        print(f"Ошибка парсинга JSON: {e}")
        return
    # Формируем нумерованный список с датой публикации или полем day (устар.)
    formatted_lines = []
    for i, item in enumerate(news_items, 1):
        title = item.get("title", "").strip()
        url = item.get("url", "").strip()
        pub = item.get("published_date")
        if not title or not url:
            continue
        if pub:
            line = f"{i}.\t{title} (published: {pub})\n{url}"
        elif day is not None:
            line = f"{i}.\t{title} (day: {day})\n{url}"
        else:
            line = f"{i}.\t{title}\n{url}"
        formatted_lines.append(line)

    # Склеиваем результаты с переводом строки между ними
    result_text = "\r\n".join(formatted_lines) + "\r\n" if formatted_lines else ""
    file_name_txt = f"{section}.txt"
    save_to_drive(file_name_txt, result_text, folder["5 news_lists"], file_format="txt") # 5 news_lists
    print(f"✅ design({section}) — успешно сохранён файл с текстом.")

In [ ]:
def design(section):
    """Render the section news list as a formatted text file via the LLM.

    LLM-based counterpart of ``design_wo_llm``. Loads ``<section>.json``
    from the ``2 4 new_lists_json`` Drive folder and asks the DeepSeek
    chat API to produce a nicely formatted numbered list using the
    ``prompt_design`` template plus the ``example`` reference layout.

    The model reply is saved as ``<section>.txt`` in the
    ``5 news_lists`` folder.

    Args:
        section: One of ``"world"``, ``"rus"``, ``"prices"``.

    Returns:
        None. Logs progress and writes the result to Drive; aborts on any
        I/O or model error.
    """
    file_name_json = f"{section}.json"
    try:
        file_id = find_file_in_drive(file_name_json, folder["2 4 new_lists_json"])
        news_list_raw = download_text_file(file_id)
    except FileNotFoundError:
        print(f"Файл {file_name_json} не найден в папке.")
        return
    except Exception as e:
        print(f"Ошибка при загрузке файла {file_name_json}: {e}")
        return
    raw_parts = [prompt_design, example, news_list_raw]
    prompt_parts = []
    for part in raw_parts:
        if isinstance(part, list):
            prompt_parts.append("\n".join(part))
        else:
            prompt_parts.append(str(part))
    prompt_text = "\n".join(prompt_parts)
    try:
        payload = {
            "model": model_lists,
            "messages": [
                {"role": "system", "content": "Отвечай лаконично и информативно. Никогда не добавляй в списки новостей источники, найденные в интернете - отбирай новости только из приложенного списка."},
                {"role": "user", "content": prompt_text}
            ],
            "temperature": 0.7
        }
    
    
    
        # payload = {
        #     "model": "sonar-pro",
        #     "messages": [
        #         {"role": "system", "content": "Отвечай лаконично и информативно. Никогда не добавляй в списки новостей источники, найденные в интернете - отбирай новости только из приложенного списка."},
        #         {"role": "user", "content": prompt_text}
        #     ],
        #     "temperature": 0.7,
        #     "disable_search": True
        # }


        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()
        result = response.json()
        choices = result.get("choices")
        if not choices or not choices[0].get("message", {}).get("content"):
            print(f"Модель не вернула ответ для '{file_name_json}'.")
            return
        assistant_text = choices[0]["message"]["content"]
        file_name_txt = f"{section}.txt"
        save_to_drive(file_name_txt, assistant_text, folder["5 news_lists"], file_format="txt") # 5 news lists 
        print(f"✅ design({section}) — успешно сохранён файл с текстом.")
    except Exception as e:
        print(f"Ошибка при вызове модели для '{file_name_json}': {e}")
        return

In [110]:
for section in ["world", "rus", "prices"]:
    try:
        design_wo_llm(section)
    except Exception as e:
        print(f"⚠️ Ошибка в design_wo_llm для '{section}': {e}. Пробую через LLM.")
        design(section)
        time.sleep(60)
#telegram_lists()

File 'world.txt' updated (ID=1mlrWAbhgz7EszpVLZjaT9H2Y5k1ZzTxd).
✅ design(world) — успешно сохранён файл с текстом.
File 'rus.txt' updated (ID=1NTAuauDQ5L94AD5SHufMi-G3v7UEwvQ5).
✅ design(rus) — успешно сохранён файл с текстом.
File 'prices.txt' updated (ID=1mf1dETTd2BOzJzGX9wRrOu8dkBmbjvuE).
✅ design(prices) — успешно сохранён файл с текстом.


In [ ]:
class NewsItem(BaseModel):
    """Pydantic schema for a single top-news item with its theme.

    Attributes:
        theme: Short topical label assigned by the LLM
            (e.g. ``"Регулирование"``, ``"Топливо"``).
        title: News headline in Russian, without the source name.
        url: Direct link to the article.
    """

    theme: str
    title: str
    url: str

def choose_top_urls(section):
    """Pick the 4 most important themes from the section list using the LLM.

    Loads ``<section>.json`` from ``2 4 new_lists_json`` and asks the
    DeepSeek chat API (with the ``top_prompts[section]`` prompt) to
    group the news into at most 4 themes with up to 3 articles each.
    Every returned item is validated against the ``NewsItem`` Pydantic
    model.

    Valid items are saved as ``<section>.json`` in the ``6 news_top``
    Drive folder. Invalid items are silently skipped with a warning.

    Args:
        section: One of ``"world"``, ``"rus"``, ``"prices"``.

    Returns:
        None. Writes a JSON file of ``{theme, title, url}`` entries to
        Drive; aborts on any I/O or model error.
    """
    file_name = f"{section}.json"
    folder_id = folder["2 4 new_lists_json"] # 2 4 
    try:
        file_id = find_file_in_drive(file_name, folder_id)
        news_list_raw = download_text_file(file_id)
    except FileNotFoundError:
        print(f"❌ Файл {file_name} не найден в папке {folder_id}.")
        return
    except Exception as e:
        print(f"❌ Ошибка при загрузке файла {file_name}: {e}")
        return
    if not news_list_raw.strip():
        print(f"❌ Файл {file_name} пустой.")
        return

    prompt_top = top_prompts.get(section, "")
    system_content = (
        "Анализируй предоставленный список новостей и выдели 4 ключевые темы. Верни список объектов с полями theme, title, url для каждой новости."
    )
#     system_content = (
#     "Анализируй предоставленный список новостей и выдели 4 ключевые темы. "
#     "Верни JSON массив объектов с полями theme, title, url для каждой новости. "
#     "Формат: [{\"theme\": \"...\", \"title\": \"...\", \"url\": \"...\"}, ...]"
# )




    prompt_text = "\n".join([str(prompt_top), news_list_raw])

    try:
        payload = {
            "model": model_bullets, 
            "messages": [
                {
                    "role": "system",
                    "content": system_content
                },
                {
                    "role": "user",
                    "content": prompt_text
                }
            ],
            "temperature": 0.2,
            "response_format": {
                "type": "json_object"
            }
        }

        # payload = {
        #     "model": "sonar-pro", 
        #     "messages": [
        #         {
        #             "role": "system",
        #             "content": system_content
        #         },
        #         {
        #             "role": "user",
        #             "content": prompt_text
        #         }
        #     ],
        #     "temperature": 0.2,
        #     "disable_search": True,
        #     "response_format": {
        #         "type": "json_schema",
        #         "json_schema": {
        #             "schema": {
        #                 "type": "array",
        #                 "items": NewsItem.model_json_schema()
        #             }
        #         }
        #     }
        # }
        
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()
        result = response.json()
        choices = result.get("choices")
        if not choices:
            print("❌ В ответе API нет поля 'choices'.")
            return
        
        content = choices[0]["message"]["content"]
        if not content:
            print("❌ Модель вернула пустой ответ.")
            return
            
        # Парсим JSON и валидируем через Pydantic
        import json
        data = json.loads(content)
        
        # Валидируем каждый элемент через Pydantic
        valid_output = []
        for item_data in data:
            try:
                news_item = NewsItem.model_validate(item_data)
                valid_output.append({
                    "theme": news_item.theme,
                    "title": news_item.title,
                    "url": news_item.url
                })
            except Exception as e:
                print(f"⚠️ Пропускаем невалидный элемент: {e}")
                continue
                
        if not valid_output:
            print("❌ Итоговый JSON пуст.")
            return
            
    except Exception as e:
        print(f"❌ Ошибка при вызове модели для '{file_name}': {e}")
        return
    
    output_folder_id = folder["6 news_top"] # 6 news top
    save_to_drive(file_name, valid_output, output_folder_id, file_format="json")
    print(f"✅ choose_top_urls({section}) — сохранён корректный JSON с новостями и темами.")


In [112]:
if datetime.today().weekday() == 3:
    choose_top_urls("world")
    time.sleep(60)
    choose_top_urls("rus")
    time.sleep(60)
    choose_top_urls("prices")

File 'world.json' updated (ID=1kTZjdIKpBN9yMQB-UjOJ8q6OjacbI9UI).
✅ choose_top_urls(world) — сохранён корректный JSON с новостями и темами.
File 'rus.json' updated (ID=146J8hjZsEsGQKtQeXGW4n-0ZNV8g7xn8).
✅ choose_top_urls(rus) — сохранён корректный JSON с новостями и темами.
File 'prices.json' updated (ID=1mm_wS0_sXiRT4eAITOT2f9V4DYs-C0cy).
✅ choose_top_urls(prices) — сохранён корректный JSON с новостями и темами.


In [114]:
datetime.today().weekday()

3

In [115]:
def read_top_urls(section, max_chars=3000):
    """Download and extract the article body for every top URL of a section.

    Reads the file produced by ``choose_top_urls`` from the
    ``6 news_top`` Drive folder (``<section>.json``). For each item it
    fetches the article HTML, runs ``extract_main_text`` to keep only
    meaningful paragraphs, and stores ``{title, url, theme, text}`` in
    ``<section>.json`` inside the ``7 news_top_texts`` folder.

    Errors on individual URLs are logged and skipped — the rest of the
    items continue to be processed.

    Args:
        section: One of ``"world"``, ``"rus"``, ``"prices"``.
        max_chars: Maximum length of the extracted article body
            (truncated on the last whitespace before the limit).

    Returns:
        None. Writes the per-section text dump to Drive.
    """
    def extract_main_text(soup, max_chars=3000, min_paragraph_len=50, max_paragraphs=5):
        """Extract a short readable summary from a parsed HTML page.

        Iterates over ``<p>`` tags and keeps the first ``max_paragraphs``
        whose text is at least ``min_paragraph_len`` characters long and
        is not obviously a cookie/subscription/advertising notice. The
        kept paragraphs are joined with spaces and truncated to
        ``max_chars`` (cut on the last whitespace boundary).

        Args:
            soup: ``BeautifulSoup`` object built from the article page.
            max_chars: Hard cap on the returned text length.
            min_paragraph_len: Minimum length of a paragraph to be kept.
            max_paragraphs: Maximum number of paragraphs to collect.

        Returns:
            A single string with the cleaned article excerpt.
        """
        paragraphs = []
        for p in soup.find_all('p'):
            text = p.get_text(" ", strip=True)
            if len(text) < min_paragraph_len:
                continue
            low = text.lower()
            # Фильтр по рекламе/подпискам
            if any(word in low for word in ["cookie", "subscribe", "advert", "реклама", "подпишитесь"]):
                continue
            paragraphs.append(text)
            if len(paragraphs) >= max_paragraphs:
                break
        combined_text = " ".join(paragraphs)
        if len(combined_text) > max_chars:
            combined_text = combined_text[:max_chars].rsplit(" ", 1)[0] + "..."
        return combined_text

    # Имя файла с топ ссылками для секции, например "world.json"
    file_name = f"{section}.json"

    # Находим ID файла в папке с топами
    file_id = find_file_in_drive(file_name, folder_id=folder["6 news_top"]) # 6 news top

    # Скачиваем содержимое файла — список словарей с title, url и темой
    json_text = download_text_file(file_id)
    try:
        items = json.loads(json_text)
    except Exception as e:
        print(f"Ошибка чтения JSON файла {file_name}: {e}")
        return

    results = []
    for item in items:
        url = item.get("url") or item.get("URL")
        title = item.get("title", "")
        theme = item.get("theme") or item.get("тема") or "undefined"
        if not url:
            continue
        try:
            resp = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
            soup = BeautifulSoup(resp.text, "html.parser")
            page_text = extract_main_text(soup, max_chars=max_chars)
            results.append({
                "title": title,
                "url": url,
                "theme": theme,
                "text": page_text
            })
        except Exception as e:
            print(f"Ошибка при обработке {url}: {e}")

    # Сохраняем результат в другую папку с текстами
    save_to_drive(
        file_name,
        results,
        my_folder=folder["7 news_top_texts"] , # 7 news_top_texts
        file_format="json"
    )
    print(f"{section}: сохранено {len(results)} ссылок с текстами.")

In [116]:
datetime.today().weekday()

3

In [117]:
if datetime.today().weekday() == 3:
    read_top_urls("world")
    read_top_urls("rus")
    read_top_urls("prices")

File 'world.json' updated (ID=15YjqU4h_7kqDeZZLktDX9j6x2JAgQkj7).
world: сохранено 12 ссылок с текстами.
File 'rus.json' updated (ID=1_-sZt4h8-bHk9gsio-OJJX9NY88OpDTs).
rus: сохранено 10 ссылок с текстами.
File 'prices.json' updated (ID=17mwqVk7EkZyzq7zHwELST5dbyq-1VFAY).
prices: сохранено 11 ссылок с текстами.


In [ ]:
def create_bullets(section):
    """Generate the final analytical bullet points for a section via the LLM.

    Loads ``<section>.json`` (top articles with their bodies) from the
    ``7 news_top_texts`` Drive folder, sends it together with the
    ``bullets_prompts[section]`` prompt to the DeepSeek chat API, and
    writes the model's plain-text reply to
    ``report_<section>.txt`` in the ``8 news_final`` folder.

    Args:
        section: One of ``"world"``, ``"rus"``, ``"prices"``.

    Returns:
        None. Logs progress and saves the bullets file to Drive; aborts
        on any I/O or model error.
    """
    list_file = f"{section}.json"
    try:
        file_id = find_file_in_drive(list_file, folder["7 news_top_texts"]) # 7 news_top_texts
        list_content = download_text_file(file_id)
    except Exception as e:
        print(f"Ошибка загрузки файла {list_file}: {e}")
        return

    # Если пришёл JSON, делаем красиво (отступы для читаемости)
    try:
        parsed_json = json.loads(list_content)
        pretty_json = json.dumps(parsed_json, ensure_ascii=False, indent=2)
    except json.JSONDecodeError:
        pretty_json = str(list_content)

    prompt_bullets = bullets_prompts.get(section, "")
    prompt_text = "\n".join([str(prompt_bullets), pretty_json])

    try:
        payload = {
            "model": model_bullets,
            "messages": [
                {"role": "system", "content": "Ты — профессиональный макроэкономический аналитик Департамента денежно-кредитной политики ЦБ РФ. Твоя задача — подготовить краткие, точные и фактологические буллиты на основе предоставленных новостей, которые будут использованы в еженедельной аналитической записке для руководства банка. Будь предельно объективен, избегай интерпретаций и сосредоточься только на фактах из предоставленных текстов. Твоя работа напрямую влияет на принятие решений по денежно-кредитной политике."},
                {"role": "user", "content": prompt_text}
            ],
            "temperature": 0.5,
        }


        # payload = {
        #     "model": "sonar-pro",
        #     "messages": [
        #         {"role": "system", "content": "Отвечай лаконично и информативно."},
        #         {"role": "user", "content": prompt_text}
        #     ],
        #     "temperature": 0.7,
        # }

        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()
        result = response.json()

        choices = result.get("choices")
        if not choices or not choices[0].get("message", {}).get("content"):
            print(f"Модель не вернула ответ для {section}.")
            return

        assistant_text = choices[0]["message"]["content"]

        file_name = f"report_{section}.txt"
        save_to_drive(file_name, assistant_text, my_folder=folder["8 news_final"], file_format="txt")

         # Сохраняем в локальный файл для архивации
        local_filename = f"report_{section}.txt"
        with open(local_filename, "w", encoding="utf-8") as f_local:
            f_local.write(assistant_text)
        print(f"Локально сохранено: {local_filename}")
        
        
        print(f"{section}: буллиты успешно записаны.")

    except Exception as e:
        print(f"Ошибка при вызове модели для {section}: {e}")
        return

In [120]:
if datetime.today().weekday() == 3:
  create_bullets("world")
  time.sleep(60)
  create_bullets("rus")
  time.sleep(60)
  create_bullets("prices")
  #telegram_bullets()

File 'report_world.txt' updated (ID=1y8DaP0KhiBAJAMjIL9Y2qLAgDoU8_z2-).
world: буллиты успешно записаны.
File 'report_rus.txt' updated (ID=1rPJuqVToe1pKfL5unXdj2Oozqu04oBtP).
rus: буллиты успешно записаны.
File 'report_prices.txt' updated (ID=1ZjJ-jm-CLJczu2KO-Q1204vIpUyeWLKv).
prices: буллиты успешно записаны.
